# Stage 0b — Bootstrap: seed the pool from your existing YOLO dataset
**Run this ONCE.** It imports your existing labelled `train/val/test` images
into the append-only pool and writes the `v1.0` manifest — your reproducible
baseline. After this, use the normal add-new-data loop (Stages 1+).

In [ ]:
import sys; sys.path.insert(0,'/workspace/lib')
import os, glob
import dataset_lib as D

In [ ]:
# --- point at your EXISTING YOLO dataset (mounted read-only is fine) ---
# Adjust this to wherever your current train/val/test dataset is mounted.
SRC_DATASET = '/existing'      # e.g. mount your YOLO26-Dataset here
DATASET_ROOT = '/dataset'
DROP_CLASS_NAME = 'Car'        # dropped (zero instances) — remapped out
BOOTSTRAP_VERSION = 'v1.0'
# the 31-class source order (with Car at 26); pool uses the 30-class scheme
SRC_NAMES = ['ErinaceusEuropaeus','SciurusCarolinensis','SciurusVulgaris','CapreolusCapreolus',
  'CervusElaphus','VulpesVulpes','MelesMeles','Person','CapraHircus','OryctolagusCuniculus',
  'ColumbaPalumbus','PhasianusColchicus','OvisAries','PasserDomesticus','DamaDama','BosTaurus',
  'FelisCatus','MartesMartes','CanisFamiliaris','CalibrationPole','AccipiterGentilis','ButeoButeo',
  'CappercaillieCock','CappercaillieHen','NumeniusArquata','NumeniusArquataChick','Car',
  'EquusCaballus','CorvusCorone','SusScrofa','MuntiacusReevesi']
# map source id -> pool (30-class) id, dropping Car
from dataset_lib import NAME_TO_ID
src_to_pool = {}
for i,nm in enumerate(SRC_NAMES):
    if nm in NAME_TO_ID: src_to_pool[i] = NAME_TO_ID[nm]
print('source classes:', len(SRC_NAMES), '-> pool classes:', len(set(src_to_pool.values())))
print('dropped:', [SRC_NAMES[i] for i in range(len(SRC_NAMES)) if i not in src_to_pool])

In [ ]:
assert os.path.isdir(SRC_DATASET), f'mount your existing dataset at {SRC_DATASET} (see docker-compose volumes)'
# gather all label files across splits
label_files = []
for split in ['train','val','test']:
    ld = os.path.join(SRC_DATASET, split, 'labels')
    if os.path.isdir(ld):
        label_files += [(split, os.path.join(ld,f)) for f in os.listdir(ld) if f.endswith('.txt')]
print(len(label_files), 'label files found across splits')

In [ ]:
# import each image+labels into the pool, remapping class ids (dropping Car)
added=skipped=remapped_boxes=dropped_boxes=0
IMG_EXTS=('.jpg','.jpeg','.png','.bmp','.webp')
for split, lf in label_files:
    stem = os.path.splitext(os.path.basename(lf))[0]
    # find the matching image
    idir = os.path.join(SRC_DATASET, split, 'images')
    src_img=None
    for e in IMG_EXTS:
        p=os.path.join(idir, stem+e)
        if os.path.exists(p): src_img=p; break
    if src_img is None: skipped+=1; continue
    # remap label lines
    out_lines=[]
    for line in open(lf):
        p=line.split()
        if len(p)<5: continue
        cid=int(float(p[0]))
        if cid not in src_to_pool: dropped_boxes+=1; continue  # e.g. Car
        out_lines.append(f"{src_to_pool[cid]} {p[1]} {p[2]} {p[3]} {p[4]}"); remapped_boxes+=1
    if not out_lines: skipped+=1; continue
    D.add_to_pool(DATASET_ROOT, src_img, out_lines); added+=1
print(f'imported {added} images | skipped {skipped} | boxes kept {remapped_boxes} dropped {dropped_boxes}')

In [ ]:
# write the v1.0 manifest — the reproducible baseline
path, man = D.write_manifest(DATASET_ROOT, BOOTSTRAP_VERSION, note='bootstrap from existing YOLO dataset')
print('manifest:', path)
print('images:', man['n_images'])
print('splits:', man['split_counts'])
print('class counts:')
for k,v in man['class_counts'].items(): print(f'  {k:<22} {v}')

Pool seeded and `v1.0` manifest written. From here, use the normal loop:
Stage 1 (ingest new images) -> Stage 2 (verify) -> Stage 3 (merge as v1.1)
-> Stage 4 (retrain) -> Stage 5 (evaluate).

To mount your existing dataset for this step, add to `docker-compose.yml`
under the pipeline service volumes:
```
      - "C:/Users/PaulF/Desktop/DeepFaune-UK/YOLO26-Dataset:/existing:ro"
```
(read-only, since we only copy out of it).